### Define a function for making per-cell data into per-cell with neighbors data.
This makes it so each cell has the information of each of it's 8 neighbors in it's row too.

In [4]:
import json

import geopandas as gpd
import gradio as gr
import h3
import numpy as np
import pandas as pd
import plotly.express as px
from shapely.geometry import shape
from pathlib import Path
import re

def build_3x3_neighborhood_df(
    df: pd.DataFrame,
    band_cols: list[str],
    cell_size: float = 20.0,
    group_cols: list[str] | None = None,
) -> pd.DataFrame:
    """
    Returns one row per center cell with all original columns plus neighbor-band features.

    - Keeps every original column from df unchanged.
    - Adds only the 8 surrounding neighbor band sets as new columns.
    - Center-cell bands remain original names (B11, B12, ...), no duplicated p0_p0 columns.
    - Uses inner joins for neighbors, so edge/missing-neighbor cells are dropped.
    """
    if group_cols is None:
        group_cols = [c for c in ["year"] if c in df.columns]

    required_cols = set(["x", "y", *band_cols, *group_cols])
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    merge_keys = [*group_cols, "x", "y"]

    # Keep all original columns as the base output
    out = df.copy()

    def offset_suffix(dx: int, dy: int) -> str:
        sx = f"{'m' if dx < 0 else 'p'}{abs(dx)}"
        sy = f"{'m' if dy < 0 else 'p'}{abs(dy)}"
        return f"{sx}_{sy}"

    for dy in (-1, 0, 1):
        for dx in (-1, 0, 1):
            # Skip center: those band columns already exist in out
            if dx == 0 and dy == 0:
                continue

            neigh = df[[*group_cols, "x", "y", *band_cols]].copy()

            # Shift neighbor coordinates into center coordinate frame
            neigh["x"] = neigh["x"] - dx * cell_size
            neigh["y"] = neigh["y"] - dy * cell_size

            suff = offset_suffix(dx, dy)
            neigh = neigh.rename(columns={c: f"{c}_{suff}" for c in band_cols})

            out = out.merge(neigh, on=merge_keys, how="inner")

    return out



### Actually use the function to convert all the data into the 3x3 format.

In [9]:
data_path = "data"
original_files = list(Path(data_path).iterdir())
output_files = [Path(str(og.parent.parent.absolute() / "data_3x3" / og.stem) + "_3x3" + str(og.suffix)) for og in original_files]

for in_file, out_file in zip(original_files, output_files):

    # Skip if the output file already exists
    if out_file.exists() and out_file.is_file():
        print(f"{out_file} already exists. Skipping...")
        continue

    print(f"Starting on input: {in_file.name}; output: {out_file.name}")
    # Load the CSV
    df = pd.read_csv(in_file, index_col=False)

    # Parse the GeoJSON
    df["geometry"] = df[".geo"].apply(lambda x: shape(json.loads(x)))

    # Example: infer band and output columns from your current schema
    band_columns = [c for c in df.columns if re.fullmatch(r"B\d+", c)]

    df_3x3 = build_3x3_neighborhood_df(
        df=df,
        band_cols=band_columns,
        cell_size=20.0,
        group_cols=["year"] if "year" in df.columns else None,
    )

    print("Original rows:", len(df))
    print("3x3 valid center rows:", len(df_3x3))
    print("Row diff:", len(df) - len(df_3x3))
    print("Feature columns:", len(df_3x3.columns))
    print(*df.columns)
    print(*df_3x3.columns)

    print(f"Saving {out_file}...")
    df_3x3.to_csv(out_file, index=False)

/home/macedmo/git/machine-learning-final-project-ws2025/dashboard/data_3x3/df_2016_mit_2021_labels_3x3.csv already exists. Skipping...
/home/macedmo/git/machine-learning-final-project-ws2025/dashboard/data_3x3/df_2018_mit_2020_labels_3x3.csv already exists. Skipping...
/home/macedmo/git/machine-learning-final-project-ws2025/dashboard/data_3x3/df_2019_mit_2020_labels_3x3.csv already exists. Skipping...
/home/macedmo/git/machine-learning-final-project-ws2025/dashboard/data_3x3/df_2017_mit_2021_labels_3x3.csv already exists. Skipping...
/home/macedmo/git/machine-learning-final-project-ws2025/dashboard/data_3x3/df_2016_mit_2020_labels_3x3.csv already exists. Skipping...
/home/macedmo/git/machine-learning-final-project-ws2025/dashboard/data_3x3/df_2018_mit_2021_labels_3x3.csv already exists. Skipping...
/home/macedmo/git/machine-learning-final-project-ws2025/dashboard/data_3x3/df_2017_mit_2020_labels_3x3.csv already exists. Skipping...
Starting on input: df_2020_mit_2021_labels.csv; output:

### Combine all of the data from all the tables for a given label year, into large tables with "delta_year" offsets from the label year.

In [27]:
data_path = "data_3x3"
label_year = 2021
files = list(Path(data_path).iterdir())

filtered_files = [file for file in files if f"mit_{label_year}_labels" in str(file)]

def extract_years(filename):
    f = str(filename.stem).split("_")
    return int(f[1]), int(f[3])

dfs = []

for file in filtered_files:
    df = pd.read_csv(file, index_col=False)
    start_year, end_year = extract_years(file)
    df.insert(0, "delta_years", end_year-start_year)
    dfs.append(df)

dfo = pd.concat(dfs)

for df in dfs:
    print(df.shape)

print(dfo.shape)

dfo.to_csv(filtered_files[0].parent / f"delta_table_{label_year}_3x3.csv", index=False)

(457363, 96)
(457363, 96)
(457363, 96)
(457363, 96)
(457363, 96)
(457363, 96)
(2744178, 96)


In [28]:
dfo["delta_years"].unique()

array([2, 3, 1, 0, 5, 4])

In [30]:
dfo.sample(5)

,delta_years,system:index,B11,B12,B2,B3,B4,B5,B6,B7,...,B8_p0_p1,B11_p1_p1,B12_p1_p1,B2_p1_p1,B3_p1_p1,B4_p1_p1,B5_p1_p1,B6_p1_p1,B7_p1_p1,B8_p1_p1
162987,4,167259,1827.5,1622.5,501.0,670.0,746.5,985.0,1533.0,1663.0,...,1655.0,1856.0,1670.0,497.0,650.0,759.0,1059.5,1375.5,1291.0,1234.0
411670,4,419603,2105.0,1870.0,1723.0,1770.0,1748.0,1809.0,1742.0,1761.0,...,1232.0,1899.0,1615.0,828.0,1009.0,1080.0,1399.0,1622.0,1845.0,1555.0
257643,2,263749,1988.5,1005.5,260.0,439.5,351.0,847.0,2531.0,3123.0,...,2519.5,1582.0,1013.0,385.5,492.0,432.5,678.0,1502.5,1875.5,1808.5
414271,5,422279,2113.0,1748.0,332.0,557.0,586.0,1120.0,2160.0,2301.0,...,2562.0,2033.0,1143.0,301.0,713.0,394.0,1181.0,3791.0,4398.0,4256.0
441391,4,449775,2325.0,2037.0,627.0,849.0,1129.0,1369.0,2771.0,3200.0,...,3315.0,2430.0,2068.0,666.0,895.0,1222.0,1425.0,2815.0,3214.0,3213.0


### Experiment with gathering data from Open Street Maps.

In [4]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
from shapely.geometry import Point
from shapely import wkt

import json
from shapely.geometry import shape

# Updated parsing function
def parse_geometry(geo_str):
    if pd.isna(geo_str) or not isinstance(geo_str, str):
        return None
    try:
        # 1. Parse the string into a Python dictionary
        geo_dict = json.loads(geo_str)
        # 2. Convert that dictionary into a Shapely object
        return shape(geo_dict)
    except:
        return None

# 1. Load your data
df = pd.read_csv("data.csv") # Assuming your sample is in a CSV
df['geometry'] = df['.geo'].apply(parse_geometry)
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:32632")

# 2. Define your target area (Nuremberg) for OSM data
place_name = "Nuremberg, Germany"

# 3. Fetch Road and Water data from OpenStreetMap
# Fetching "primary" and "secondary" roads as they are major change drivers
roads = ox.features_from_place(place_name, tags={"highway": ["primary", "secondary", "tertiary"]})
water = ox.features_from_place(place_name, tags={"natural": "water", "waterway": True})

# 4. Project OSM data to match your UTM 32N (EPSG:32632) for meter-based distance
roads = roads.to_crs("EPSG:32632")
water = water.to_crs("EPSG:32632")

# 5. Create a "Built-up" reference from your OWN 2016 data
# This is crucial for predicting change based on existing urban footprints [cite: 104]
existing_urban = gdf[gdf['built_up'] > 0.5].unary_union

def get_min_dist(point, target_gdf):
    """Calculates minimum distance from a point to any object in a GeoDataFrame."""
    return target_gdf.distance(point).min()

# 6. Calculate proximity features
print("Calculating distances... this may take a moment.")
gdf['dist_to_road'] = gdf.geometry.centroid.apply(lambda x: get_min_dist(x, roads))
gdf['dist_to_water'] = gdf.geometry.centroid.apply(lambda x: get_min_dist(x, water))
gdf['dist_to_builtup'] = gdf.geometry.centroid.apply(lambda x: x.distance(existing_urban))

print(gdf[['dist_to_road', 'dist_to_water', 'dist_to_builtup']].head())

/tmp/ipykernel_293076/3264745948.py:41: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  existing_urban = gdf[gdf['built_up'] > 0.5].unary_union


Calculating distances... this may take a moment.
   dist_to_road  dist_to_water  dist_to_builtup
0    349.033473      74.740731       496.588361
1    360.386728      93.039641       505.371151
2    372.468552     111.922174       514.781507
3    385.210399     131.136460       524.785671
4    398.548968     150.555535       535.350353


In [5]:
gdf.to_csv("data_with_extras.csv")

### Experiment with loading ML Models

In [1]:
import json
import pickle
from pathlib import Path

import geopandas as gpd
import gradio as gr
import numpy as np
import pandas as pd
import plotly.express as px
from shapely.geometry import box, shape

def load_prediction_model(model_path="artifacts/xgboost_multioutput.pkl"):
    model_file = Path(model_path)
    if not model_file.exists():
        return None

    with model_file.open("rb") as f:
        return pickle.load(f)

model = load_prediction_model()

In [4]:
model.estimators_

[XGBRegressor(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.678024823943575, device='cuda',
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None, feature_weights=None,
              gamma=0.4287949729173641, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.033161675945190996,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=8, max_leaves=None,
              min_child_weight=8, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=226, n_jobs=-1,
              num_parallel_tree=None, ...),
 XGBRegressor(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.678024823943575, device='cuda',
 

In [6]:

def load_data_from_csv(csv_path="data.csv", cell_size_m=200):
    # 1. Load CSV data
    df = pd.read_csv(csv_path)

    # 2. Parse geometry from GeoJSON strings
    df["geometry"] = df[".geo"].apply(lambda x: shape(json.loads(x)))

    # 3. Build GeoDataFrame in projected CRS (meters)
    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:32632")

    # Aggregate 20x20m cells into larger display cells.
    gdf["grid_x"] = (np.floor(gdf["x"] / cell_size_m) * cell_size_m).astype(int)
    gdf["grid_y"] = (np.floor(gdf["y"] / cell_size_m) * cell_size_m).astype(int)

    # agg_df = (
    #     gdf.groupby(["grid_x", "grid_y"], as_index=False)
    #     .agg({**{column: "mean" for column in class_cols}, "cell_id": "count"})
    #     .rename(columns={"cell_id": "cell_count"})
    # )

    numeric_columns = gdf.select_dtypes(include=[np.number]).columns.tolist()
    agg_columns = {
        column: "mean"
        for column in numeric_columns
        if column not in {"grid_x", "grid_y"}
    }
    agg_df = gdf.groupby(["delta_years", "grid_x", "grid_y"], as_index=False).agg(
        agg_columns
    )

    agg_df["geometry"] = agg_df.apply(
        lambda row: box(
            row["grid_x"],
            row["grid_y"],
            row["grid_x"] + cell_size_m,
            row["grid_y"] + cell_size_m,
        ),
        axis=1,
    )

    gdf = gpd.GeoDataFrame(agg_df, geometry="geometry", crs="EPSG:32632")

    # 4. Reproject to latitude/longitude
    gdf = gdf.to_crs("EPSG:4326")

    # 5. Create unique IDs for Plotly linking
    gdf["Hexagon_ID"] = gdf.index

    # Add vegetation target label
    gdf["vegetation"] = gdf[["tree_cover", "cropland", "grassland"]].sum(axis=1)

    # Add engineered features
    gdf["NDVI"] = (gdf["B8"] - gdf["B4"]) / (gdf["B8"] + gdf["B4"] + 1e-8)
    gdf["EVI2"] = 2.5 * (
        (gdf["B8"] - gdf["B4"]) / (gdf["B8"] + 2.4 * gdf["B4"] + 1 + 1e-8)
    )
    gdf["SAVI"] = ((gdf["B8"] - gdf["B4"]) * 1.5) / (gdf["B8"] + gdf["B4"] + 0.5 + 1e-8)
    gdf["NDBI"] = (gdf["B11"] - gdf["B8"]) / (gdf["B11"] + gdf["B8"] + 1e-8)
    gdf["NDWI"] = (gdf["B3"] - gdf["B8"]) / (gdf["B3"] + gdf["B8"] + 1e-8)
    gdf["MNDWI"] = (gdf["B3"] - gdf["B11"]) / (gdf["B3"] + gdf["B11"] + 1e-8)

    # 6. Convert to GeoJSON format expected by Plotly
    plotly_geojson = json.loads(gdf.geometry.to_json())
    return gdf, plotly_geojson

In [7]:
load_data_from_csv("data_3x3/delta_table_2021_3x3.csv")

KeyboardInterrupt: 